In [ ]:
from datascience import *
import numpy as np

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')

import plotly.express as px

from scipy.stats import norm
from scipy.stats import t

## Central Limit Theorem

In [ ]:
united = Table.read_table('/home/jovyan/shared/Data/united.csv')

delays = united.column('Delay')
delay_mean = np.mean(delays)
delay_sd = np.std(delays)

def one_sample_mean(sample_size):
    """ Takes a sample from the population of flights and computes its mean"""
    sampled_flights = united.sample(sample_size)
    return np.mean(sampled_flights.column('Delay'))

def ten_thousand_sample_means(sample_size):
    means = make_array()
    for i in np.arange(10000):
        mean = one_sample_mean(sample_size)
        means = np.append(means, mean)
    return means

"""Empirical distribution of random sample means"""

def plot_and_summarize_sample_means(sample_size):
    sample_means = ten_thousand_sample_means(sample_size)
    sample_means_tbl = Table().with_column('Sample Means', sample_means)
    
    # Print some information about the distribution of the sample means
    print("Sample size: ", sample_size)
    print("Population mean:", delay_mean)
    print("Average of sample means: ", np.mean(sample_means))
    print("Population SD:", delay_sd)
    print("SD of sample means:", np.std(sample_means))

    # Plot a histogram of the sample means
    sample_means_tbl.hist(bins=30)
    plots.xlabel('Sample Means')
    plots.title('Sample Size ' + str(sample_size))
    
    # Plot normal curve over histogram
    x = np.arange(
        delay_mean - 4*delay_sd/np.sqrt(sample_size), 
        delay_mean + 4*delay_sd/np.sqrt(sample_size),
        0.1
    )
    plots.plot(x, 
           norm.pdf(
               x, 
               delay_mean, 
               delay_sd/np.sqrt(sample_size)
           )
          );

In [ ]:
plot_and_summarize_sample_means(400)

## Swain vs. Alabama - Normal Distribution

In [ ]:
population_proportions = make_array(.26, .74)

def panel_proportion():
    return sample_proportions(100, population_proportions).item(0)

panels = make_array()

num_simulations = 100000

for i in np.arange(num_simulations):
    new_panel = panel_proportion()
    panels = np.append(panels, new_panel)


Table().with_column('Proportion of Black Men on Panel of 100', panels).hist(bins=np.arange(0.055,.40, 0.01))
plots.axvline(0.08, color = 'red', lw = 2);

In [ ]:
SD_pop = (0.26 * (1 - 0.26)) ** 0.5
SD_panels_formula = SD_pop / (100) ** 0.5

x = np.arange(0.055,.40, 0.001)
plots.plot(x, norm.pdf(x, 0.26, SD_panels_formula));
plots.axvline(0.08, color = 'red', lw = 2);
print("p-value from the normal distribution = ", norm.cdf(0.08, loc = 0.26, scale =  SD_panels_formula))

## Smile Leniency Effect - t-distribution

In [ ]:
Smiles = Table.read_table('/home/jovyan/shared/Data/Smiles.csv')

def difference_of_means(table, outcome, group_label):
    """Takes: name of table, column label of binary categorical variable,
    column label of group-label variable
    Returns: Difference of proportions of the two groups"""
    
    #table with the two relevant columns
    reduced = table.select(outcome, group_label)  
    
    # table containing counts for each group
    summary_table = reduced.group(group_label, np.mean)
    
    return summary_table.column(1).item(0) - summary_table.column(1).item(1)

obs_diff_mean = difference_of_means(Smiles, 'Leniency', 'Group')

def one_simulated_difference(table, outcome, group_label):
    """Takes: name of table, column label of binary cateogorcial variable,
    column label of group-label variable
    Returns: Difference of proportions of the two groups after shuffling labels"""
    
    # array of shuffled labels
    shuffled_labels = table.sample(with_replacement = False).column(group_label)
    
    # table of outcome variable and shuffled labels
    shuffled_table = table.select(outcome).with_column(
        'Shuffled Label', shuffled_labels)
    
    return difference_of_means(shuffled_table, outcome, 'Shuffled Label')  

differences = make_array()

for i in np.arange(10000):
    differences = np.append(differences, one_simulated_difference(Smiles, 'Leniency', 'Group'))


Table().with_column('Difference Between Group Means', differences).hist(bins = 20)
plots.axvline(obs_diff_mean, color = 'red', lw=2);

p_value = np.mean(differences <= obs_diff_mean)
p_value

In [ ]:
sd_smile = np.std(Smiles.where('Group', 'smile').column('Leniency'))
sd_neutral = np.std(Smiles.where('Group', 'neutral').column('Leniency'))

n_smile = Smiles.where('Group', 'smile').num_rows
n_neutral = Smiles.where('Group', 'neutral').num_rows

sd_diff = np.sqrt(sd_smile**2 / n_smile + sd_neutral**2/n_neutral)

t_score = (obs_diff_mean - 0) / sd_diff

x1 = np.arange(-4, 4, 0.01)
plots.plot(x1, t.pdf(x1, min(n_smile, n_neutral) - 1));
x2 = np.arange(-4, t_score, 0.01)
plots.fill_between(x2, t.pdf(x2, min(n_smile, n_neutral) - 1), color = 'red');

p_value = t.cdf(t_score, min(n_smile, n_neutral) - 1)
p_value

## Data Feminism Book Reviews

In [ ]:
resample_star_props = make_array()

for i in np.arange(10000):
    resample_star_props = np.append(resample_star_props, sample_proportions(1599, make_array(1351/1599, 1 - 1351/1599)).item(0))

left_end_df = percentile(5, resample_star_props)
right_end_df = percentile(95, resample_star_props)
    
Table().with_column("Proportion of Star Ratings at 4 or Above in Resamples", 
                     resample_star_props).hist("Proportion of Star Ratings at 4 or Above in Resamples", bins = 15)
plots.plot([left_end_df,right_end_df], [0, 0], color='gold', lw=10);

left_end_df,right_end_df

In [ ]:
mean_normal = 1351/1599
sd_normal = np.sqrt((1351/1599 * (1 - 1351/1599)) / 1599)

x3 = np.arange(min(resample_star_props),max(resample_star_props), 0.0001)
plots.plot(x3, norm.pdf(x3, mean_normal, sd_normal));

z_star = norm.ppf(0.95)

left_end_df_norm = mean_normal - z_star * sd_normal
right_end_df_norm = mean_normal + z_star * sd_normal

x4 = np.arange(left_end_df_norm, right_end_df_norm, 0.001)
plots.fill_between(x4, norm.pdf(x4, mean_normal, sd_normal), color = 'red');

left_end_df_norm, right_end_df_norm

## Linear regression

In [ ]:
def standard_units(x):
    """Converts an array x to standard units"""
    return (x - np.mean(x)) / np.std(x)

def correlation(t, x, y):
    x_su = standard_units(t.column(x))
    y_su = standard_units(t.column(y))
    return np.mean(x_su * y_su)

def slope(t, x, y):
    """t is a table; x and y are column labels"""
    r = correlation(t, x, y)
    slope_xy = r * np.std(t.column(y)) / np.std(t.column(x))
    return slope_xy

def intercept(t, x, y):
    """t is a table; x and y are column labels"""
    slope_xy = slope(t, x, y)
    intercept_xy = np.average(t.column(y)) - slope_xy * np.average(t.column(x))
    return intercept_xy

def fitted_values(t, x, y):
    """Return the height of the regression line at each x value"""
    a = slope(t, x, y)
    b = intercept(t, x, y)
    return a * t.column(x) + b

def residual(t, x, y):
    """Return the residuals of the regression line at each x value."""
    return t.column(y) - fitted_values(t, x, y)

def residual_plot(t, x, y):
    """displays a residual plot for the least squares regression line"""
    Table().with_columns(
            'fitted values', fitted_values(t, x, y),
            'residuals', residual(t, x, y)
    ).scatter('fitted values', 'residuals', color='r')
    plots.axhline(0, color='darkblue', lw=4)
    plots.title('Residual Plot')

def prediction_at(t, x, y, x_value):
    """ return the least squares regression prediction for x_value"""
    return slope(t, x, y) * x_value + intercept(t, x, y)

def add_regression_to_table(t, x, y):
    """ Returns a new table with 'Fitted value' and 'Residual' columns added """
    t_with_extra_cols = t.with_columns(
        'Fitted value', fitted_values(t, x, y),
        'Residual', residuals(t, x, y)
    )
    return t_with_extra_cols

In [ ]:
demographics = Table.read_table('/home/jovyan/shared/Data/district_demographics2016.csv')
demographics.scatter('College%', 'Median Income', fit_line = True)

a = correlation(demographics, 'College%', 'Median Income')
b = slope(demographics, 'College%', 'Median Income')
c = intercept(demographics, 'College%', 'Median Income')

a, b, c

In [ ]:
def demographics_rmse(any_slope, any_intercept):
    x = demographics.column('College%')
    y = demographics.column('Median Income')
    estimate = any_slope*x + any_intercept
    return (np.mean((y - estimate) ** 2)) ** 0.5

minimize(demographics_rmse)

In [ ]:
residual_plot(demographics, 'College%', 'Median Income')

In [ ]:
def demographics_quad_rmse(a, b, c):
    x = demographics.column('College%')
    y = demographics.column('Median Income')
    estimate = a*x**2 + b*x + c
    return (np.mean((y - estimate) ** 2)) ** 0.5

best_quad = minimize(demographics_quad_rmse)

demographics.scatter('College%', 'Median Income')

x5 = np.arange(5, 75, 0.5)
y = best_quad.item(0)*(x5**2) + best_quad.item(1)*x5 + best_quad.item(2)

plots.plot(x5, y, color = 'red', lw=2);

In [ ]:
best_quad.item(0)*(25**2) + best_quad.item(1)*25 + best_quad.item(2)

In [ ]:
px.scatter_3d(
    demographics.to_df(), 
    x='College%', 
    y='Percent voting for Clinton', 
    z='Median Income'
).update_traces({'marker':{'size': 3}}
               ).update_layout(width=800, height=600)

In [ ]:
def demographics_mult_rmse(a, b, c):
    x = demographics.column('College%')
    z = demographics.column('Percent voting for Clinton')
    y = demographics.column('Median Income')
    
    estimate = a*x + b*z + c
    
    return np.mean((y - estimate)**2)**0.5

coefficients = minimize(demographics_mult_rmse)

coefficients.item(0)*25 + coefficients.item(1)*40 + coefficients.item(2)

In [ ]:
# back to line with one predictor
prediction_at(demographics, 'College%', 'Median Income', 25)

In [ ]:
linear_rmse = demographics_rmse(minimize(demographics_rmse).item(0), minimize(demographics_rmse).item(1))
quad_rmse = demographics_quad_rmse(best_quad.item(0), best_quad.item(1), best_quad.item(2))
mult_rmse = demographics_mult_rmse(coefficients.item(0), coefficients.item(1), coefficients.item(2))

linear_rmse, quad_rmse, mult_rmse

In [ ]:
def bootstrap_prediction(t, x, y, new_x, repetitions=1000):

    # Bootstrap the scatter, predict, collect
    predictions = make_array()
    for i in np.arange(repetitions):
        resample = t.sample()
        predicted_y = prediction_at(resample, x, y, new_x)
        predictions = np.append(predictions, predicted_y)

    # Find the ends of the approximate 95% prediction interval
    left = percentile(2.5, predictions)
    right = percentile(97.5, predictions)

    # Display results
    Table().with_column('Prediction', predictions).hist(bins=20)
    plots.xlabel('predictions at x='+str(new_x))
    plots.plot([left, right], [0, 0], color='yellow', lw=8);
    print('Approximate 95%-confidence interval for height of true line:')
    print(left, right, '(width =', right - left, ')') 

bootstrap_prediction(demographics, 'College%', 'Median Income', 30)

In [ ]:
def bootstrap_slope(t, x, y, repetitions=1000):
    
    # Bootstrap the scatter, find the slope, collect
    slopes = make_array()
    for i in np.arange(repetitions):
        bootstrap_sample = t.sample()
        bootstrap_slope = slope(bootstrap_sample, x, y)
        slopes = np.append(slopes, bootstrap_slope)
    
    # Find the endpoints of the 95% confidence interval for the true slope
    left = percentile(2.5, slopes)
    right = percentile(97.5, slopes)
    
    # Slope of the regression line from the original sample
    observed_slope = slope(t, x, y)
    
    # Display results
    Table().with_column('Bootstrap Slopes', slopes).hist(bins=20)
    plots.plot(make_array(left, right), make_array(0, 0), color='yellow', lw=8);
    print('Slope of regression line:', observed_slope)
    print('Approximate 95%-confidence interval for the slope of the true line:')
    print(left, 'to', right)

In [ ]:
bootstrap_slope(demographics, 'Percent voting for Clinton', 'Median Income')